In [1]:
!nvidia-smi

Mon Aug 23 15:42:18 2021       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.57.02    Driver Version: 470.57.02    CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:01:00.0 Off |                  N/A |
|  0%   39C    P8    10W / 280W |     15MiB / 11177MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2020 NVIDIA Corporation
Built on Mon_Oct_12_20:09:46_PDT_2020
Cuda compilation tools, release 11.1, V11.1.105
Build cuda_11.1.TC455_06.29190527_0


In [3]:
import os
import random
from datetime import datetime
import time

In [4]:
os.chdir('..')
# os.chdir('..')

In [5]:
os.getcwd()

'/home/hmatsuya/workspace/Shogi/dlcobra'

In [6]:
pip install -e .

Obtaining file:///home/hmatsuya/workspace/Shogi/dlcobra
  Attempting uninstall: dlshogi
    Found existing installation: dlshogi 0.0.6
    Uninstalling dlshogi-0.0.6:
      Successfully uninstalled dlshogi-0.0.6
  Running setup.py develop for dlshogi
Note: you may need to restart the kernel to use updated packages.


In [7]:
# os.listdir('//corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe')
# os.listdir('/mnt/corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe')

In [8]:
os.getcwd()

'/home/hmatsuya/workspace/Shogi/dlcobra'

In [9]:
os.chdir('dlshogi')

In [10]:
#os.mkdir('../model')

In [11]:
#os.mkdir('../log')

## Test critic

In [ ]:
# resnet8_dropout
# # advantage: no value
# val_lambda: 0
# tanh
# polic_coef: 0.5

chunk_size = 38 * 20 * 1000 * 1000
activation='C:\\Anaconda3\\Scripts\\activate.bat'
test='/home/hmatsuya/workspace/Shogi/data/dl_data/hcpe/floodgate_teacher_uniq-test-01'

#! wsl ssh hmatsuya@corgi ls -lh /home/hmatsuya/workspace/Shogi/data/apery_teacher/

# resume = '-r ../model/state-2019'
resume = ''
teacher_dir = '/home/hmatsuya/workspace/Shogi/data/apery_teacher/shuffled'
filelist = os.listdir(teacher_dir)
hcpelist = list(filter(lambda f: '.hcpe' in f, filelist))
files = []
project = "test_critic"
test_name = "result_critic"
name = f'{test_name}'
run_id = f'{name}.{time.time()}'
i = 0
if i == 0:
    resume = ''
else:
    resume = f'-m ../model/model-{name}-{i-1} -r ../model/checkpoint-{name}-{i-1}.pth'

for file in random.sample(hcpelist, len(hcpelist)):
    
    files.append(file)
    if len(files) < 10:
        continue
    
    teacher=' '.join([f'{teacher_dir}/{f}' for f in files])
    
    model=f'../model/model-{name}-{i}'
    checkpoint=f'../model/checkpoint-{name}-{i}.pth'
    log=f'../log/{name}.txt'
    ! python -u train.py {teacher} {test} --model {model} --checkpoint {checkpoint} {resume} --use_result_critic --val_lambda 0.3 --lr 0.001 --weight_decay 0.00001 --use_average --use_evalfix --use_swa --log {log} --project {project} --run_id {run_id}

    files.clear()
    resume = f'-r {checkpoint} -m {model}'
    i += 1


2021/08/19 04:26:42	INFO	network resnet10_swish
2021/08/19 04:26:42	INFO	batchsize=1024
2021/08/19 04:26:42	INFO	lr=0.001
2021/08/19 04:26:42	INFO	weight_decay=1e-05
2021/08/19 04:26:42	INFO	val_lambda=0.3
wandb: Currently logged in as: hmatsuya (use `wandb login --relogin` to force relogin)
wandb: Tracking run with wandb version 0.12.0
wandb: Syncing run result_critic
wandb:  View project at https://wandb.ai/hmatsuya/test_critic
wandb:  View run at https://wandb.ai/hmatsuya/test_critic/runs/result_critic.1629314800.877809
wandb: Run data is saved locally in /home/hmatsuya/workspace/Shogi/dlcobra/dlshogi/wandb/run-20210819_042644-result_critic.1629314800.877809
wandb: Run `wandb offline` to turn off syncing.

2021/08/19 04:26:47	INFO	use swa(swa_start_epoch=1, swa_freq=250, swa_n_avr=10)
2021/08/19 04:26:47	INFO	use evalfix
2021/08/19 04:26:47	INFO	temperature=1.0
2021/08/19 04:26:47	INFO	optimizer SGD (Parameter Group 0 dampening: 0 lr: 0.001 momentum: 0.9 nesterov: True weight_decay:

In [5]:
# Convert to ONNX
!python ../dlshogi/convert_model_to_onnx.py --network resnet10_swish ../model/model-result_critic-15 ../model/model-result_critic-15.onnx

graph(%input1 : Float(*, 62, 9, 9, strides=[5022, 81, 9, 1], requires_grad=0, device=cuda:0),
      %input2 : Float(*, 57, 9, 9, strides=[4617, 81, 9, 1], requires_grad=0, device=cuda:0),
      %l1_1_1.weight : Float(192, 62, 3, 3, strides=[558, 9, 3, 1], requires_grad=1, device=cuda:0),
      %l1_1_2.weight : Float(192, 62, 1, 1, strides=[62, 1, 1, 1], requires_grad=1, device=cuda:0),
      %l1_2.weight : Float(192, 57, 1, 1, strides=[57, 1, 1, 1], requires_grad=1, device=cuda:0),
      %l22.weight : Float(27, 192, 1, 1, strides=[192, 1, 1, 1], requires_grad=1, device=cuda:0),
      %l22_2.bias : Float(2187, strides=[1], requires_grad=1, device=cuda:0),
      %l23_v.weight : Float(256, 2187, strides=[2187, 1], requires_grad=1, device=cuda:0),
      %l23_v.bias : Float(256, strides=[1], requires_grad=1, device=cuda:0),
      %l24_v.weight : Float(1, 256, strides=[256, 1], requires_grad=1, device=cuda:0),
      %l24_v.bias : Float(1, strides=[1], requires_grad=1, device=cuda:0),
      %